## <small>
Copyright (c) 2017-21 Andrew Glassner

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
</small>



# Deep Learning: A Visual Approach
## by Andrew Glassner, https://glassner.com
### Order: https://nostarch.com/deep-learning-visual-approach
### GitHub: https://github.com/blueberrymusic
------

### What's in this notebook

This notebook is provided as a “behind-the-scenes” look at code used to make some of the figures in this chapter. It is cleaned up a bit from the original code that I hacked together, and is only lightly commented. I wrote the code to be easy to interpret and understand, even for those who are new to Python. I tried never to be clever or even more efficient at the cost of being harder to understand. The code is in Python3, using the versions of libraries as of April 2021.

This notebook may contain additional code to create models and images not in the book. That material is included here to demonstrate additional techniques.

Note that I've included the output cells in this saved notebook, but Jupyter doesn't save the variables or data that were used to generate them. To recreate any cell's output, evaluate all the cells from the start up to that cell. A convenient way to experiment is to first choose "Restart & Run All" from the Kernel menu, so that everything's been defined and is up to date. Then you can experiment using the variables, data, functions, and other stuff defined in this notebook.

## Chapter 23: Creative Applications - Notebook 2: Loss Visualization

### About this code:
This notebook is adapted from
https://github.com/fchollet/deep-learning-with-python-notebooks/blob/master/8.3-neural-style-transfer.ipynb
by François Chollet.

See License C in LICENSE.txt

In [67]:
import os
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.optimize import fmin_l_bfgs_b
from skimage.io import imread, imsave

import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import vgg16
from tensorflow.keras import backend as K

# Use channels_last (height, width, channels)
K.set_image_data_format("channels_last")

In [68]:
# Workaround for Keras issues on Mac computers (you can comment this
# out if you're not on a Mac, or not having problems)
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [69]:
# Because we will be making many images, and we don't
# want to fill up the notebook with one after another,
# we use imread and imsave from skimage.io instead
# of a custom File_Helper.

import os

save_files = False  # set True if you want files actually written

# Base directories
input_data_directory = os.path.join(os.getcwd(), "input_data")
output_data_directory = os.path.join(os.getcwd(), "saved_output")

# Ensure directories exist
os.makedirs(input_data_directory, exist_ok=True)
os.makedirs(output_data_directory, exist_ok=True)

print("Input data directory :", input_data_directory)
print("Output data directory:", output_data_directory)
print("save_files =", save_files)

Input data directory : /content/input_data
Output data directory: /content/saved_output
save_files = False


In [70]:
from tensorflow.keras.preprocessing.image import load_img

def get_target_size(target_image_path, target_height):
    """Return the size (width, height) for resizing, preserving aspect ratio."""
    width, height = load_img(target_image_path).size
    img_width = int(width * target_height / height)
    return (img_width, target_height)

In [71]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import vgg16
import numpy as np

def preprocess_image(image_path, img_height, img_width):
    """Process an image using VGG16 conventions."""
    img = load_img(image_path, target_size=(img_height, img_width))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = vgg16.preprocess_input(img)
    return img


def deprocess_image(x):
    """Undo VGG16 image conventions."""
    x = np.array(x, copy=True)
    # Remove zero-center by mean pixel (VGG16 convention)
    x[:, :, 0] += 103.939
    x[:, :, 1] += 116.779
    x[:, :, 2] += 123.68
    # Convert from BGR to RGB
    x = x[:, :, ::-1]
    x = np.clip(x, 0, 255).astype("uint8")
    return x

In [72]:
from tensorflow.keras.applications import vgg16

def get_VGG16():
    """Get VGG16 model without top dense layers, with ImageNet weights."""
    # Use the default Keras Input; we will feed tensors to this model later.
    model = vgg16.VGG16(
        weights="imagenet",
        include_top=False,
    )
    return model

In [73]:
from tensorflow.keras import backend as K

# Content loss

def content_loss(base, combination):
    """Content loss: encourage similar high-level representation."""
    return K.sum(K.square(combination - base))

In [74]:
from tensorflow.keras import backend as K

# Style loss

def gram_matrix(x):
    """Compute a Gram matrix: G_ij = sum_k F_ik * F_jk."""
    # Reorder to (channels, height, width) then flatten spatial dims
    features = K.batch_flatten(K.permute_dimensions(x, (2, 0, 1)))
    gram = K.dot(features, K.transpose(features))
    return gram


def style_loss(style, combination, img_height, img_width):
    """Style loss: match Gram matrices between style and combination images."""
    S = gram_matrix(style)
    C = gram_matrix(combination)
    channels = 3
    size = img_height * img_width
    return K.sum(K.square(S - C)) / (4.0 * (channels ** 2) * (size ** 2))

In [75]:
from tensorflow.keras import backend as K

# Total variation loss

def total_variation_loss(x, img_height, img_width):
    """Total variation loss: encourages spatial smoothness."""
    a = K.square(
        x[:, :img_height - 1, :img_width - 1, :] -
        x[:, 1:, :img_width - 1, :]
    )
    b = K.square(
        x[:, :img_height - 1, :img_width - 1, :] -
        x[:, :img_height - 1, 1:, :]
    )
    return K.sum(K.pow(a + b, 1.25))

In [76]:
# An object to hold both loss and gradients, so we can calculate
# both at once and cache them, returning either one when asked.

class Evaluator(object):
    def __init__(self, img_height, img_width, fetch_loss_and_grads):
        self.loss_value = None
        self.grad_values = None
        self.img_height = img_height
        self.img_width = img_width
        self.fetch_loss_and_grads = fetch_loss_and_grads

    def loss(self, x):
        assert self.loss_value is None
        x = x.reshape((1, self.img_height, self.img_width, 3))
        loss_value, grad_values = self.fetch_loss_and_grads([x])
        self.loss_value = loss_value
        self.grad_values = grad_values.flatten().astype("float64")
        return self.loss_value

    def grads(self, x):
        assert self.loss_value is not None
        grad_values = np.copy(self.grad_values)
        self.loss_value = None
        self.grad_values = None
        return grad_values

In [86]:
import time
import os

import numpy as np
from scipy.optimize import fmin_l_bfgs_b
from skimage.io import imsave

import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import vgg16
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input # Explicitly import Input
from tensorflow.keras.models import Model # Explicitly import Model

def make_synthetic_image(
    target_image_path,
    style_reference_image_path,
    output_prefix,
    target_height,
    iterations,
    content_layer,
    style_layers,
    total_variation_weight,
    style_weight,
    content_weight,
    random_seed,
):
    # Compute target size with same aspect ratio as target image
    img_width, img_height = get_target_size(target_image_path, target_height)

    # Define symbolic Input layers for the three images
    # All inputs to the VGG model must be symbolic Keras tensors.
    target_image_input = Input(shape=(img_height, img_width, 3), name='target_image_input')
    style_reference_image_input = Input(shape=(img_height, img_width, 3), name='style_reference_image_input')
    combination_image_input = Input(shape=(img_height, img_width, 3), name='combination_image_input')

    # Concatenate the three symbolic inputs into a single batch for VGG16
    vgg_input_batch = K.concatenate(
        [target_image_input, style_reference_image_input, combination_image_input], axis=0
    )

    # Build VGG16 model with this concatenated symbolic batch as input
    model = vgg16.VGG16(
        input_tensor=vgg_input_batch,
        weights="imagenet",
        include_top=False,
    )
    print("Model loaded.")

    # Dict mapping layer names to activation tensors (these are symbolic tensors)
    outputs_dict = {layer.name: layer.output for layer in model.layers}

    # Loss calculation (symbolic Keras tensor)
    loss = K.variable(0.0)

    # Content loss
    layer_features = outputs_dict[content_layer]
    target_image_features = layer_features[0, :, :, :]
    combination_features = layer_features[2, :, :, :]
    loss = loss + content_weight * content_loss(target_image_features, combination_features)

    # Style loss
    if style_layers:
        for layer_name in style_layers:
            layer_features = outputs_dict[layer_name]
            style_reference_features = layer_features[1, :, :, :]
            combination_features = layer_features[2, :, :, :]
            sl = style_loss(
                style_reference_features,
                combination_features,
                img_height,
                img_width,
            )
            loss = loss + (style_weight / float(len(style_layers))) * sl

    # Total variation loss (applied to the symbolic combination_image_input)
    loss = loss + total_variation_weight * total_variation_loss(
        combination_image_input, img_height, img_width
    )

    # Gradients of the loss wrt the symbolic combination_image_input
    grads = K.gradients(loss, combination_image_input)[0]

    # Create a Keras functional Model that takes the three input tensors
    # and outputs the computed loss and gradients. This is needed for `model.predict`.
    loss_and_grads_model = Model(
        inputs=[target_image_input, style_reference_image_input, combination_image_input],
        outputs=[loss, grads]
    )

    # The function that the Evaluator will use to fetch concrete loss and gradients
    def fetch_loss_and_grads_concrete(combination_image_data_np):
        # Preprocess target and style images to concrete numpy arrays
        target_data_np = preprocess_image(target_image_path, img_height, img_width)
        style_data_np = preprocess_image(style_reference_image_path, img_height, img_width)

        # Use the functional model to predict loss and gradients from concrete numpy inputs
        loss_value, grad_values = loss_and_grads_model.predict(
            [target_data_np, style_data_np, combination_image_data_np]
        )
        return loss_value, grad_values


    evaluator = Evaluator(img_height, img_width, fetch_loss_and_grads_concrete)

    # Initialize x with random noise (same shape as preprocessed target)
    x_initial = preprocess_image(target_image_path, img_height, img_width)
    np.random.seed(random_seed)
    x = np.random.uniform(low=-128, high=128, size=x_initial.shape).astype("float32")
    x = x.flatten()

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_prefix), exist_ok=True)

    for i in range(iterations):
        print("Start of iteration", i)
        start_time = time.time()

        x, min_val, info = fmin_l_bfgs_b(
            evaluator.loss, x, fprime=evaluator.grads, maxfun=20
        )
        print("Current loss value:", min_val)

        # Save current generated image
        img = x.copy().reshape((img_height, img_width, 3))
        img = deprocess_image(img)
        fname = (
            f"{output_prefix}-iteration-{i}"
            f"-sw-{style_weight:.5f}"
            f"-cw-{content_weight:.5f}"
            f"-tvw-{total_variation_weight:.6f}.png"
        )
        imsave(fname, img)
        end_time = time.time()
        print("Image saved as", fname)
        print("Iteration %d completed in %ds" % (i, end_time - start_time))


In [78]:
def make_style_visualizations():
    vgg16_layers_list = [
        "block1_conv1", "block1_conv2",
        "block2_conv1", "block2_conv2",
        "block3_conv1", "block3_conv2", "block3_conv3",
        "block4_conv1", "block4_conv2", "block4_conv3",
        "block5_conv1", "block5_conv2", "block5_conv3",
    ]

    # Use plain paths instead of file_helper
    target_image_path = os.path.join(
        input_data_directory, "waters-3038803_1280-crop.jpg"
    )
    style_reference_image_path = os.path.join(
        input_data_directory, "HR-Self-Portrait-1907-Picasso.jpg"
    )

    target_height = 200
    num_iterations = 50
    content_layer = "block4_conv1"   # irrelevant for style
    total_variation_weight = 1e-3
    style_weight = 1.0
    content_weight = 0.0

    # Ensure output directory exists
    os.makedirs(output_data_directory, exist_ok=True)

    for i in range(len(vgg16_layers_list)):
        style_layers = vgg16_layers_list[: i + 1]
        random_seed = 42 + i
        output_prefix = os.path.join(
            output_data_directory, "style-to-" + style_layers[-1]
        )
        print("Starting style viz of layers", style_layers)
        make_synthetic_image(
            target_image_path=target_image_path,
            style_reference_image_path=style_reference_image_path,
            output_prefix=output_prefix,
            target_height=target_height,
            iterations=num_iterations,
            content_layer=content_layer,
            style_layers=style_layers,
            total_variation_weight=total_variation_weight,
            style_weight=style_weight,
            content_weight=content_weight,
            random_seed=random_seed,
        )

In [79]:
def make_single_layer_style_visualizations():
    vgg16_layers_list = [
        "block1_conv1", "block1_conv2",
        "block2_conv1", "block2_conv2",
        "block3_conv1", "block3_conv2", "block3_conv3",
        "block4_conv1", "block4_conv2", "block4_conv3",
        "block5_conv1", "block5_conv2", "block5_conv3",
    ]

    # Use plain paths instead of file_helper
    target_image_path = os.path.join(
        input_data_directory, "waters-3038803_1280-crop.jpg"
    )  # irrelevant for style viz
    style_reference_image_path = os.path.join(
        input_data_directory, "HR-Self-Portrait-1907-Picasso.jpg"
    )

    target_height = 200
    num_iterations = 50
    content_layer = "block4_conv1"   # irrelevant for style
    total_variation_weight = 1e-3
    style_weight = 1.0
    content_weight = 0.0

    os.makedirs(output_data_directory, exist_ok=True)

    # skip the first layer because we got it when we did the sets
    for i in range(1, len(vgg16_layers_list)):
        style_layers = [vgg16_layers_list[i]]
        random_seed = 424 + i
        output_prefix = os.path.join(
            output_data_directory, "style-only-" + style_layers[-1]
        )
        print("Starting style viz of layers", style_layers)
        make_synthetic_image(
            target_image_path=target_image_path,
            style_reference_image_path=style_reference_image_path,
            output_prefix=output_prefix,
            target_height=target_height,
            iterations=num_iterations,
            content_layer=content_layer,
            style_layers=style_layers,
            total_variation_weight=total_variation_weight,
            style_weight=style_weight,
            content_weight=content_weight,
            random_seed=random_seed,
        )

In [80]:
def make_content_visualizations():
    vgg16_layers_list = [
        "block1_conv1", "block1_conv2",
        "block2_conv1", "block2_conv2",
        "block3_conv1", "block3_conv2", "block3_conv3",
        "block4_conv1", "block4_conv2", "block4_conv3",
        "block5_conv1",
        "block5_conv2", "block5_conv3",
    ]

    # Use plain paths instead of file_helper
    target_image_path = os.path.join(
        input_data_directory, "waters-3038803_1280-crop.jpg"
    )
    style_reference_image_path = os.path.join(
        input_data_directory, "HR-Self-Portrait-1907-Picasso.jpg"
    )  # irrelevant for content viz

    style_layers = []  # irrelevant for content viz
    target_height = 200
    num_iterations = 50
    total_variation_weight = 1e-3
    style_weight = 0.0
    content_weight = 1.0

    os.makedirs(output_data_directory, exist_ok=True)

    for i, content_layer in enumerate(vgg16_layers_list):
        random_seed = 4242 + i
        print("Starting content viz of layer", content_layer)
        output_prefix = os.path.join(output_data_directory, "content-" + content_layer)

        make_synthetic_image(
            target_image_path=target_image_path,
            style_reference_image_path=style_reference_image_path,
            output_prefix=output_prefix,
            target_height=target_height,
            iterations=num_iterations,
            content_layer=content_layer,
            style_layers=style_layers,
            total_variation_weight=total_variation_weight,
            style_weight=style_weight,
            content_weight=content_weight,
            random_seed=random_seed,
        )

In [81]:
def make_style_and_content_visualizations():
    # make_style_visualizations()
    # make_content_visualizations()
    make_single_layer_style_visualizations()

In [82]:
import os
import requests

# These must match the variables you already defined earlier
input_data_directory = os.path.join(os.getcwd(), "input_data")
output_data_directory = os.path.join(os.getcwd(), "saved_output")

os.makedirs(input_data_directory, exist_ok=True)
os.makedirs(output_data_directory, exist_ok=True)

# Cat image (Pexels, royalty-free)
cat_url = "https://images.pexels.com/photos/617278/pexels-photo-617278.jpeg"
cat_dest = os.path.join(input_data_directory, "waters-3038803_1280-crop.jpg")

print("Downloading cat image to:", cat_dest)
r = requests.get(cat_url)
r.raise_for_status()
with open(cat_dest, "wb") as f:
    f.write(r.content)
print("Done.")

Done.


In [83]:
import os
import requests

input_data_directory = os.path.join(os.getcwd(), "input_data")
os.makedirs(input_data_directory, exist_ok=True)

# Painting‑like image from Pexels (royalty‑free)
style_url = "https://images.pexels.com/photos/102127/pexels-photo-102127.jpeg"
style_dest = os.path.join(input_data_directory, "HR-Self-Portrait-1907-Picasso.jpg")

print("Downloading style image to:", style_dest)
r = requests.get(style_url)
r.raise_for_status()
with open(style_dest, "wb") as f:
    f.write(r.content)
print("Done.")

Done.


In [84]:
import os
print(os.path.exists(os.path.join(input_data_directory, "waters-3038803_1280-crop.jpg")))
print(os.path.exists(os.path.join(input_data_directory, "HR-Self-Portrait-1907-Picasso.jpg")))

True
True
